In [ ]:
from pathlib import Path
import numpy as np
import rasterio
from tqdm import tqdm
import random
import shutil

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PATCH_SIZE = 128
STRIDE = 16
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
def read_band(city_dir, band):
    path = list(city_dir.glob(f"*.{band}.tif"))[0]
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32)

def normalize_percentile(img, p_low=2, p_high=98):
    low, high = np.percentile(img, (p_low, p_high))
    img = np.clip(img, low, high)
    return (img - low) / (high - low + 1e-8)

def normalize_thermal(img):
    img = np.clip(img, 300, 320)
    return (img - 300) / 20.0

def safe_index(a, b):
    return (a - b) / (a + b + 1e-8)

In [ ]:
def build_input_target(city_dir):
    b2 = read_band(city_dir, "SR_B2")   # Blue
    b3 = read_band(city_dir, "SR_B3")   # Green
    b4 = read_band(city_dir, "SR_B4")   # Red
    b5 = read_band(city_dir, "SR_B5")   # NIR
    b6 = read_band(city_dir, "SR_B6")   # SWIR1
    b7 = read_band(city_dir, "SR_B7")   # SWIR2
    b10 = read_band(city_dir, "ST_B10") # Thermal

    h = min(b2.shape[0], b3.shape[0], b4.shape[0], b5.shape[0], b6.shape[0], b7.shape[0], b10.shape[0])
    w = min(b2.shape[1], b3.shape[1], b4.shape[1], b5.shape[1], b6.shape[1], b7.shape[1], b10.shape[1])

    b2, b3, b4, b5, b6, b7, b10 = [
        x[:h, :w] for x in [b2, b3, b4, b5, b6, b7, b10]
    ]

    ndvi = safe_index(b5, b4)
    ndwi = safe_index(b3, b5)
    ndbi = safe_index(b6, b5)

    thermal = normalize_thermal(b10)
    ndvi = np.clip((ndvi + 1) / 2, 0, 1)
    ndwi = np.clip((ndwi + 1) / 2, 0, 1)
    ndbi = np.clip((ndbi + 1) / 2, 0, 1)
    swir2 = normalize_percentile(b7)

    input_tensor = np.stack([thermal, ndvi, ndwi, ndbi, swir2], axis=-1).astype(np.float32)

    r = normalize_percentile(b4)
    g = normalize_percentile(b3)
    b = normalize_percentile(b2)

    target_tensor = np.stack([r, g, b], axis=-1).astype(np.float32)

    return input_tensor, target_tensor

In [ ]:
def extract_patches(input_tensor, target_tensor, patch_size=128, stride=16):
    h, w, _ = input_tensor.shape
    input_patches = []
    target_patches = []

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            inp = input_tensor[y:y+patch_size, x:x+patch_size, :]
            tgt = target_tensor[y:y+patch_size, x:x+patch_size, :]

            if np.isnan(inp).any() or np.isnan(tgt).any():
                continue

            if inp.std() < 0.01 or tgt.std() < 0.01:
                continue

            input_patches.append(inp)
            target_patches.append(tgt)

    return input_patches, target_patches

In [ ]:
all_inputs = []
all_targets = []
all_meta = []

city_dirs = sorted([p for p in RAW_DIR.iterdir() if p.is_dir()])

for city_dir in tqdm(city_dirs):
    inp, tgt = build_input_target(city_dir)
    input_patches, target_patches = extract_patches(inp, tgt, PATCH_SIZE, STRIDE)

    for i in range(len(input_patches)):
        all_inputs.append(input_patches[i])
        all_targets.append(target_patches[i])
        all_meta.append(city_dir.name)

print("Total patches:", len(all_inputs))
print("Input shape:", all_inputs[0].shape)
print("Target shape:", all_targets[0].shape)

In [ ]:
indices = list(range(len(all_inputs)))
random.shuffle(indices)

n = len(indices)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

splits = {
    "train": indices[:train_end],
    "val": indices[train_end:val_end],
    "test": indices[val_end:]
}

for split, idxs in splits.items():
    print(split, len(idxs))

In [ ]:
for split in ["train", "val", "test"]:
    for folder in ["inputs", "targets"]:
        path = PROCESSED_DIR / split / folder
        if path.exists():
            shutil.rmtree(path)
        path.mkdir(parents=True, exist_ok=True)

In [ ]:
for split, idxs in splits.items():
    input_dir = PROCESSED_DIR / split / "inputs"
    target_dir = PROCESSED_DIR / split / "targets"

    for count, idx in enumerate(tqdm(idxs, desc=f"Saving {split}")):
        city = all_meta[idx]
        file_name = f"{city}_patch_{count:05d}.npy"

        np.save(input_dir / file_name, all_inputs[idx])
        np.save(target_dir / file_name, all_targets[idx])

print("Patch dataset saved successfully.")